In [1]:
import seaborn as sns
import pandas as pd

taxis = sns.load_dataset("taxis")
taxis.shape

(6433, 14)

In [2]:
taxis["pickup"] = pd.to_datetime(taxis["pickup"])
taxis["dropoff"] = pd.to_datetime(taxis["dropoff"])
taxis.dtypes

pickup             datetime64[us]
dropoff            datetime64[us]
passengers                  int64
distance                  float64
fare                      float64
tip                       float64
tolls                     float64
total                     float64
color                         str
payment                       str
pickup_zone                   str
dropoff_zone                  str
pickup_borough                str
dropoff_borough               str
dtype: object

In [3]:
taxis["pickup"].min()
taxis["pickup"].max()

Timestamp('2019-03-31 23:43:45')

In [4]:
taxis["jour"] = taxis["pickup"].dt.date

par_jour = taxis.groupby("jour").agg(
    nb_courses=("total", "count"),
    chiffre_affaires=("total", "sum"),
).round(2)
par_jour

,nb_courses,chiffre_affaires
jour,,
2019-02-28,1,6.30
2019-03-01,241,4213.83
2019-03-02,198,3319.02
2019-03-03,169,3027.32
2019-03-04,171,3269.08
2019-03-05,228,4377.38
2019-03-06,257,4673.61
2019-03-07,218,4115.70
2019-03-08,235,4524.21


In [5]:
par_jour["nb_courses"].idxmax()

datetime.date(2019, 3, 14)

In [6]:
courses_par_heure = taxis["pickup"].dt.hour.value_counts().sort_index()
courses_par_heure

pickup
0     205
1     110
2     101
3      67
4      57
5      51
6     142
7     221
8     313
9     319
10    325
11    295
12    334
13    316
14    360
15    330
16    336
17    388
18    417
19    406
20    368
21    355
22    321
23    296
Name: count, dtype: int64

In [7]:
courses_par_heure.idxmax()

np.int32(18)

In [8]:
pourboire_par_jour = (
    taxis.groupby(taxis["pickup"].dt.day_name())["tip"]
    .mean()
    .round(2)
    .sort_values(ascending=False)
)
pourboire_par_jour

pickup
Wednesday    2.09
Thursday     2.06
Monday       2.03
Tuesday      2.00
Friday       1.98
Sunday       1.91
Saturday     1.82
Name: tip, dtype: float64

In [9]:
taxis["duree_min"] = (taxis["dropoff"] - taxis["pickup"]).dt.total_seconds() / 60
taxis["duree_min"].describe()

count    6433.000000
mean       14.349617
std        11.643892
min         0.000000
25%         6.500000
50%        10.900000
75%        18.516667
max       107.666667
Name: duree_min, dtype: float64

In [10]:
(taxis["duree_min"] < 0).sum()
(taxis["duree_min"] < 1).sum()
(taxis["duree_min"] > 60).sum()

np.int64(50)

In [11]:
taxis_valides = taxis[taxis["duree_min"] >= 1]
len(taxis), len(taxis_valides)

(6433, 6361)

### Courses de moins d'une minute écartées des analyses de durée

| Quoi | Pourquoi | Lignes |
|---|---|---|
| Courses < 1 minute exclues | Durées irréalistes (certaines à 0 min, vitesses de centaines de mph) | 72 |
| Courses > 60 minutes conservées | Plausibles (max 108 min, longues distances) | 50 examinées, 0 retirée |